# AI-Powered Smart Fleet Optimisation Platform
## Bus Passenger Count Prediction — Model Evaluation Notebook

**Course:** Artificial Intelligence and Machine Learning (24CSE0316)  
**Target:** `Passenger_Count`  
**Engineered feature:** `Historical_Stop_Mean`

This notebook implements a leakage-aware chronological regression workflow. It compares models using the same single engineered feature, runs a separate multiple-linear-regression experiment with legitimate pre-trip variables, performs time-series cross-validation and creates figures and exportable results.

**Important:** The dataset is synthetic. Metrics describe performance on this dataset and should not be interpreted as validated real-world ridership performance.

## 0. Course-handout alignment

The notebook operationalises the course handout's topics:

- CLO02: ML workflow and Python libraries
- CLO03: preprocessing, feature engineering and regression modelling
- CLO06: Python implementation of ML algorithms
- Regression: simple, multiple and polynomial regression
- Evaluation: MAE, MSE, RMSE and R²
- Validation: chronological train/test split and time-series cross-validation
- Diagnostics: underfitting, overfitting and bias-variance behaviour

KNN, Decision Tree, Random Forest and SVM are evaluated through their **regression variants**, because `Passenger_Count` is a numerical target. Classification-only metrics such as accuracy, precision, recall, F1-score and confusion matrix are not appropriate for this regression task.

In [ ]:
# 1. Install/import libraries
!pip -q install pandas numpy matplotlib seaborn scikit-learn joblib openpyxl

import os, glob, json, warnings, zipfile, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    mean_absolute_percentage_error
)
from sklearn.inspection import permutation_importance
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titleweight"] = "bold"
RANDOM_STATE = 42

print("Libraries imported successfully.")

## 2. Upload the cleaned dataset

Upload `FINAL_CLEANED_ENCODED_DATASET.csv` when prompted. The notebook checks the required columns and prints the actual dataset dimensions. It does not silently assume that the file has the expected number of rows.

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Please upload the cleaned CSV dataset.")

# Prefer the expected filename if it is among the uploaded files.
preferred = [f for f in csv_files if "FINAL_CLEANED_ENCODED_DATASET" in f.upper()]
DATA_PATH = preferred[0] if preferred else csv_files[0]

df_raw = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df_raw.shape)
display(df_raw.head())

In [ ]:
# 3. Dataset validation and structural inspection

TARGET = "Passenger_Count"
STOP_COL = "Stop_ID"
DATE_COL = "Date"
TIME_COL = "Scheduled_Time"

required = [TARGET, STOP_COL, DATE_COL, TIME_COL]
missing_required = [c for c in required if c not in df_raw.columns]
if missing_required:
    raise ValueError(f"Required columns missing from dataset: {missing_required}")

print("Dataset shape:", df_raw.shape)
print("Duplicate rows:", df_raw.duplicated().sum())
print("Missing values:", int(df_raw.isna().sum().sum()))
print("Unique stops:", df_raw[STOP_COL].nunique())

display(pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
    "unique_values": df_raw.nunique(dropna=True)
}).sort_values("missing_count", ascending=False))

display(df_raw.describe(include="all").T)

In [ ]:
# 4. Parse dates and scheduled times; establish chronological order

df = df_raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce")

# Handles values such as "08:30:00", "08:30" and pandas-compatible time strings.
df["_scheduled_delta"] = pd.to_timedelta(df[TIME_COL].astype(str), errors="coerce")
df["_timestamp"] = df[DATE_COL] + df["_scheduled_delta"]

bad_time_rows = df["_timestamp"].isna().sum()
print("Rows with invalid date/time:", bad_time_rows)

# Rows without a valid chronological timestamp cannot be safely placed in a chronological split.
df = df.dropna(subset=["_timestamp", TARGET, STOP_COL]).copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()

sort_cols = ["_timestamp"]
if "Trip_ID" in df.columns:
    sort_cols.append("Trip_ID")

df = df.sort_values(sort_cols).reset_index(drop=True)

print("Usable rows after required-field validation:", len(df))
print("Date range:", df["_timestamp"].min(), "to", df["_timestamp"].max())
print("Target minimum / maximum:", df[TARGET].min(), "/", df[TARGET].max())
display(df[[DATE_COL, TIME_COL, "_timestamp", STOP_COL, TARGET]].head())

## 5. Exploratory Data Analysis — figures

The following plots document target distribution, outliers, demand over time, stop-level variation and the relationship between historical stop demand and passenger count. They are also useful report figures.

In [ ]:
# Figure 1: Passenger-count distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(df[TARGET], bins=30, kde=True, ax=axes[0])
axes[0].set_title("Figure 1A. Passenger Count Distribution")
axes[0].set_xlabel("Passenger_Count")
sns.boxplot(x=df[TARGET], ax=axes[1])
axes[1].set_title("Figure 1B. Passenger Count — Outlier Overview")
axes[1].set_xlabel("Passenger_Count")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Passenger counts over chronological time
daily = df.set_index("_timestamp")[TARGET].resample("D").agg(["mean", "median", "count"])
fig, ax1 = plt.subplots(figsize=(15, 5))
ax1.plot(daily.index, daily["mean"], label="Daily mean passenger count")
ax1.plot(daily.index, daily["median"], label="Daily median passenger count", alpha=0.8)
ax1.set_title("Figure 2. Passenger Demand Over Time")
ax1.set_xlabel("Date")
ax1.set_ylabel("Passenger_Count")
ax1.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(15, 4))
ax.bar(daily.index, daily["count"], width=1.0)
ax.set_title("Figure 3. Number of Records per Day")
ax.set_xlabel("Date")
ax.set_ylabel("Trip records")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 4: Stop-level demand distribution
stop_stats = (
    df.groupby(STOP_COL)[TARGET]
      .agg(["mean", "median", "std", "count"])
      .sort_values("mean", ascending=False)
)
display(stop_stats.head(15).round(2))

fig, ax = plt.subplots(figsize=(14, 6))
stop_stats.head(15)["mean"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Figure 4. Top 15 Stops by Average Passenger Count")
ax.set_xlabel("Average Passenger_Count")
ax.set_ylabel("Stop_ID")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 6))
stop_stats.tail(15)["mean"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Figure 5. Bottom 15 Stops by Average Passenger Count")
ax.set_xlabel("Average Passenger_Count")
ax.set_ylabel("Stop_ID")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 6: Day/weather exploratory plots, when the relevant columns exist
day_cols = [c for c in df.columns if c.lower() == "day"]
weather_cols = [c for c in df.columns if c.lower() == "weather"]

if day_cols:
    tmp = df.groupby(day_cols[0])[TARGET].mean().sort_values()
    ax = tmp.plot(kind="bar", figsize=(10, 5))
    ax.set_title("Figure 6A. Average Passenger Count by Day")
    ax.set_ylabel("Average Passenger_Count")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
elif any(c.startswith("Day_") for c in df.columns):
    dcols = [c for c in df.columns if c.startswith("Day_")]
    day_means = {c.replace("Day_", ""): df.loc[df[c] == 1, TARGET].mean() for c in dcols}
    pd.Series(day_means).sort_values().plot(kind="bar", figsize=(10, 5), title="Figure 6A. Average Passenger Count by Day")
    plt.ylabel("Average Passenger_Count")
    plt.tight_layout()
    plt.show()

if weather_cols:
    tmp = df.groupby(weather_cols[0])[TARGET].mean().sort_values()
    tmp.plot(kind="bar", figsize=(9, 5), title="Figure 6B. Average Passenger Count by Weather")
    plt.ylabel("Average Passenger_Count")
    plt.tight_layout()
    plt.show()
elif any(c.startswith("Weather_") for c in df.columns):
    wcols = [c for c in df.columns if c.startswith("Weather_")]
    weather_means = {c.replace("Weather_", ""): df.loc[df[c] == 1, TARGET].mean() for c in wcols}
    pd.Series(weather_means).sort_values().plot(kind="bar", figsize=(9, 5), title="Figure 6B. Average Passenger Count by Weather")
    plt.ylabel("Average Passenger_Count")
    plt.tight_layout()
    plt.show()
else:
    print("No original Day/Weather or recognisable one-hot columns found; skipping those plots.")

## 6. Chronological train-test split

We use the project design's chronological 80:20 holdout: earlier trips for training and later trips for testing. This is more appropriate than randomly shuffling trips when the objective is to predict future passenger demand.

In [ ]:
split_index = int(len(df) * 0.80)
train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

print("Training rows:", len(train))
print("Testing rows:", len(test))
print("Training period:", train["_timestamp"].min(), "to", train["_timestamp"].max())
print("Testing period :", test["_timestamp"].min(), "to", test["_timestamp"].max())

assert train["_timestamp"].max() <= test["_timestamp"].min(), "Chronological split check failed."

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train["_timestamp"], train[TARGET], label="Training period", alpha=0.7)
ax.plot(test["_timestamp"], test[TARGET], label="Testing period", alpha=0.7)
ax.axvline(test["_timestamp"].min(), linestyle="--", label="Train/test boundary")
ax.set_title("Figure 7. Chronological Train-Test Split")
ax.set_xlabel("Timestamp")
ax.set_ylabel("Passenger_Count")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Leakage-aware engineering of `Historical_Stop_Mean`

For a training row, the feature is calculated from **earlier observations only**. If a stop has no earlier observation, the fallback is the expanding global mean from earlier rows. The first chronological row has no past target history, so it is excluded from training rather than filled with a mean calculated from future training targets.

For test rows, stop means are calculated from the training period only. An unseen test stop receives the overall training mean. `Occupancy_Percentage`, `Occupancy_Category`, `Actual_Time` and `Delay_Minutes` are not predictors.

In [ ]:
# 8. Build Historical_Stop_Mean without using future target values

train = train.copy()
test = test.copy()

# Stop-specific mean of prior observations only.
train["_stop_prior_mean"] = (
    train.groupby(STOP_COL, sort=False)[TARGET]
         .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)

# Global mean of all prior observations only.
train["_global_prior_mean"] = train[TARGET].shift(1).expanding(min_periods=1).mean()

# Use stop history when available; otherwise use global past history.
train["Historical_Stop_Mean"] = train["_stop_prior_mean"].combine_first(
    train["_global_prior_mean"]
)

# The first chronological record has no prior passenger-count history.
first_row_without_history = int(train["Historical_Stop_Mean"].isna().sum())
train = train.dropna(subset=["Historical_Stop_Mean"]).copy()

# Test feature: all training observations are in the past relative to the test period.
training_stop_means = train.groupby(STOP_COL)[TARGET].mean()
training_global_mean = train[TARGET].mean()

test["Historical_Stop_Mean"] = (
    test[STOP_COL].map(training_stop_means).fillna(training_global_mean)
)

print("Training rows excluded due to no prior history:", first_row_without_history)
print("Training feature missing:", train["Historical_Stop_Mean"].isna().sum())
print("Testing feature missing:", test["Historical_Stop_Mean"].isna().sum())
print("Training shape:", train.shape, "| Testing shape:", test.shape)

display(train[[STOP_COL, TARGET, "Historical_Stop_Mean"]].head(10))
display(test[[STOP_COL, TARGET, "Historical_Stop_Mean"]].head(10))

# Leakage checks
assert train["Historical_Stop_Mean"].notna().all()
assert test["Historical_Stop_Mean"].notna().all()
assert "Occupancy_Percentage" not in ["Historical_Stop_Mean"]

In [ ]:
# Figure 8: Engineered feature distribution and relationship with target
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(train["Historical_Stop_Mean"], kde=True, bins=30, ax=axes[0])
axes[0].set_title("Figure 8A. Historical_Stop_Mean Distribution")
sns.scatterplot(
    data=train.sample(min(1500, len(train)), random_state=RANDOM_STATE),
    x="Historical_Stop_Mean", y=TARGET, alpha=0.45, ax=axes[1]
)
axes[1].set_title("Figure 8B. Historical Stop Mean vs Actual Passenger Count")
plt.tight_layout()
plt.show()

print("Correlation:", train[["Historical_Stop_Mean", TARGET]].corr().iloc[0, 1].round(4))

## 9. Experiment A — fair single-feature algorithm comparison

Every model below receives the exact same input: `Historical_Stop_Mean`. The chronological training and test sets are identical for all algorithms.

The compared models are:
1. Simple Linear Regression
2. Polynomial Regression (degree 2)
3. KNN Regression
4. Decision Tree Regression
5. Random Forest Regression
6. Support Vector Regression (RBF)

Scaling is fitted inside the KNN and SVR pipelines so the test set does not influence scaling.

In [ ]:
X_train = train[["Historical_Stop_Mean"]]
X_test = test[["Historical_Stop_Mean"]]
y_train = train[TARGET].astype(float)
y_test = test[TARGET].astype(float)

single_feature_models = {
    "Simple Linear Regression": LinearRegression(),
    "Polynomial Regression (degree 2)": make_pipeline(
        PolynomialFeatures(degree=2, include_bias=False),
        LinearRegression()
    ),
    "KNN Regression (k=5)": make_pipeline(
        StandardScaler(),
        KNeighborsRegressor(n_neighbors=5)
    ),
    "Decision Tree Regression": DecisionTreeRegressor(
        max_depth=5, random_state=RANDOM_STATE
    ),
    "Random Forest Regression": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=3,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "Support Vector Regression (RBF)": make_pipeline(
        StandardScaler(),
        SVR(C=10, epsilon=0.1, kernel="rbf")
    )
}

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_true, y_pred)
    }

results = []
predictions = {}
fitted_models = {}

for name, estimator in single_feature_models.items():
    model = clone(estimator)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    fitted_models[name] = model
    predictions[name] = pred

    row = {"Model": name, **regression_metrics(y_test, pred)}
    results.append(row)

comparison_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(comparison_df.round(4))

In [ ]:
# Figure 9: Compare all four required regression metrics
metric_specs = [
    ("MAE", "Lower is better"),
    ("MSE", "Lower is better"),
    ("RMSE", "Lower is better"),
    ("R2", "Higher is better")
]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, (metric, direction) in zip(axes.flat, metric_specs):
    plot_df = comparison_df.sort_values(metric, ascending=(metric != "R2"))
    sns.barplot(data=plot_df, x=metric, y="Model", ax=ax)
    ax.set_title(f"Figure 9. Model Comparison — {metric} ({direction})")
    ax.set_xlabel(metric)
    ax.set_ylabel("")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 10: Actual vs predicted plots for every single-feature model
n_models = len(predictions)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (name, pred) in zip(axes.flat, predictions.items()):
    sns.scatterplot(x=y_test, y=pred, alpha=0.45, s=25, ax=ax)
    lo = min(float(y_test.min()), float(np.min(pred)))
    hi = max(float(y_test.max()), float(np.max(pred)))
    ax.plot([lo, hi], [lo, hi], linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Actual Passenger_Count")
    ax.set_ylabel("Predicted Passenger_Count")
fig.suptitle("Figure 10. Actual vs Predicted Passenger Counts", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 11: Residual diagnostics for every single-feature model
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for ax, (name, pred) in zip(axes.flat, predictions.items()):
    residuals = y_test.to_numpy() - np.asarray(pred)
    sns.scatterplot(x=pred, y=residuals, alpha=0.4, s=22, ax=ax)
    ax.axhline(0, linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Predicted Passenger_Count")
    ax.set_ylabel("Residual (actual - predicted)")
fig.suptitle("Figure 11. Residuals vs Predicted Values", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (name, pred) in zip(axes.flat, predictions.items()):
    residuals = y_test.to_numpy() - np.asarray(pred)
    sns.histplot(residuals, kde=True, bins=25, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Residual")
fig.suptitle("Figure 12. Residual Error Distributions", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 10. Time-series cross-validation

The course handout includes cross-validation, overfitting, underfitting and bias-variance trade-off. We use `TimeSeriesSplit` rather than shuffled K-fold validation, preserving temporal order.

Each validation fold occurs after its training fold. The reported mean and standard deviation of RMSE show both average validation performance and its variation across time periods.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
cv_rows = []

for name, estimator in single_feature_models.items():
    scores = cross_validate(
        clone(estimator),
        train[["Historical_Stop_Mean"]],
        train[TARGET].astype(float),
        cv=tscv,
        scoring={
            "mae": "neg_mean_absolute_error",
            "mse": "neg_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1,
        return_train_score=True
    )

    fold_rmse = np.sqrt(-scores["test_mse"])
    cv_rows.append({
        "Model": name,
        "CV_RMSE_Mean": fold_rmse.mean(),
        "CV_RMSE_Std": fold_rmse.std(),
        "CV_MAE_Mean": (-scores["test_mae"]).mean(),
        "CV_R2_Mean": scores["test_r2"].mean(),
        "Train_RMSE_Mean": np.sqrt(-scores["train_mse"]).mean()
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV_RMSE_Mean").reset_index(drop=True)
display(cv_df.round(4))

In [ ]:
# Figure 13: Cross-validation RMSE with standard deviation
plot_cv = cv_df.sort_values("CV_RMSE_Mean")
fig, ax = plt.subplots(figsize=(13, 6))
ax.barh(
    plot_cv["Model"], plot_cv["CV_RMSE_Mean"],
    xerr=plot_cv["CV_RMSE_Std"], capsize=4
)
ax.set_title("Figure 13. Time-Series Cross-Validation RMSE (mean ± std)")
ax.set_xlabel("RMSE")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

# Figure 14: Training vs cross-validation RMSE — diagnostic for generalisation gap
diag = cv_df.copy()
diag["Generalisation_Gap"] = diag["CV_RMSE_Mean"] - diag["Train_RMSE_Mean"]
display(diag[["Model", "Train_RMSE_Mean", "CV_RMSE_Mean", "Generalisation_Gap"]].round(4))

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(diag))
width = 0.36
ax.bar(x - width/2, diag["Train_RMSE_Mean"], width, label="Training RMSE")
ax.bar(x + width/2, diag["CV_RMSE_Mean"], width, label="CV RMSE")
ax.set_xticks(x)
ax.set_xticklabels(diag["Model"], rotation=25, ha="right")
ax.set_title("Figure 14. Training vs Cross-Validation RMSE")
ax.set_ylabel("RMSE")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Experiment B — Multiple Linear Regression

The course handout explicitly includes Multiple Linear Regression. It cannot be meaningfully demonstrated by using only one predictor, so this separate experiment adds legitimate information available before a trip:

- `Historical_Stop_Mean`
- `Bus_Capacity`, if present
- `Peak_Hour`, if present
- Scheduled hour and minute derived from the scheduled timestamp
- Day-of-week and month derived from the date
- Existing one-hot `Day_*` and `Weather_*` columns, if present

We explicitly exclude target-derived occupancy fields and post-trip fields such as actual time and delay. This is a separate feature-set experiment and must not be mixed into Experiment A's single-feature leaderboard.

In [ ]:
# Create pre-trip calendar/time features.
for frame in (train, test):
    frame["Scheduled_Hour"] = frame["_timestamp"].dt.hour
    frame["Scheduled_Minute"] = frame["_timestamp"].dt.minute
    frame["Day_Of_Week_Num"] = frame["_timestamp"].dt.dayofweek
    frame["Month_Num"] = frame["_timestamp"].dt.month

# Candidate predictors: use only columns that actually exist.
multi_candidates = [
    "Historical_Stop_Mean", "Bus_Capacity", "Peak_Hour",
    "Scheduled_Hour", "Scheduled_Minute", "Day_Of_Week_Num", "Month_Num"
]
multi_candidates += [c for c in df.columns if c.startswith("Day_")]
multi_candidates += [c for c in df.columns if c.startswith("Weather_")]

# Strict exclusions: never use target, target-derived or post-trip columns.
forbidden_tokens = [
    "Passenger_Count", "Occupancy", "Actual_Time", "Delay",
    "Avg_Passenger", "Max_Passenger", "Passenger_Demand",
    "Avg_Occupancy", "Avg_Delay"
]
multi_features = []
for c in dict.fromkeys(multi_candidates):
    if c in train.columns and not any(token.lower() in c.lower() for token in forbidden_tokens):
        if c not in multi_features:
            multi_features.append(c)

# Ensure target engineered feature remains included.
multi_features = [c for c in multi_features if c != TARGET]
print("Multiple regression predictors:")
print(multi_features)

if len(multi_features) < 2:
    print("Not enough additional pre-trip predictors found. Multiple regression experiment cannot run.")
else:
    X_train_multi = train[multi_features].copy()
    X_test_multi = test[multi_features].copy()

    # Treat booleans as numeric; convert object columns to categorical.
    for c in multi_features:
        if X_train_multi[c].dtype == "bool":
            X_train_multi[c] = X_train_multi[c].astype(int)
            X_test_multi[c] = X_test_multi[c].astype(int)

    numeric_features = X_train_multi.select_dtypes(include=np.number).columns.tolist()
    categorical_features = [c for c in multi_features if c not in numeric_features]

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    transformers = []
    if numeric_features:
        transformers.append(("num", numeric_pipe, numeric_features))
    if categorical_features:
        transformers.append(("cat", categorical_pipe, categorical_features))

    multi_preprocessor = ColumnTransformer(transformers=transformers)
    multiple_linear_model = Pipeline([
        ("preprocess", multi_preprocessor),
        ("regressor", LinearRegression())
    ])

    multiple_linear_model.fit(X_train_multi, y_train)
    multi_pred = multiple_linear_model.predict(X_test_multi)
    multi_metrics = regression_metrics(y_test, multi_pred)

    multi_results_df = pd.DataFrame([{
        "Experiment": "Multiple Linear Regression — pre-trip predictors",
        "Features": ", ".join(multi_features),
        **multi_metrics
    }])
    display(multi_results_df.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(x=y_test, y=multi_pred, alpha=0.45, ax=axes[0])
    lo = min(float(y_test.min()), float(np.min(multi_pred)))
    hi = max(float(y_test.max()), float(np.max(multi_pred)))
    axes[0].plot([lo, hi], [lo, hi], linestyle="--")
    axes[0].set_title("Figure 15A. Multiple Linear Regression — Actual vs Predicted")
    axes[0].set_xlabel("Actual Passenger_Count")
    axes[0].set_ylabel("Predicted Passenger_Count")

    multi_residuals = y_test.to_numpy() - multi_pred
    sns.histplot(multi_residuals, kde=True, ax=axes[1])
    axes[1].set_title("Figure 15B. Multiple Linear Regression — Residuals")
    axes[1].set_xlabel("Residual")
    plt.tight_layout()
    plt.show()

## 12. Polynomial relationship visualisation

This plot helps explain why Polynomial Regression was tested. It overlays the fitted degree-2 curve and the simple linear regression line on the same training observations.

In [ ]:
linear_fit = fitted_models["Simple Linear Regression"]
poly_fit = fitted_models["Polynomial Regression (degree 2)"]

x_grid = pd.DataFrame({
    "Historical_Stop_Mean": np.linspace(
        train["Historical_Stop_Mean"].min(),
        train["Historical_Stop_Mean"].max(),
        300
    )
})

plt.figure(figsize=(12, 6))
sample = train.sample(min(1200, len(train)), random_state=RANDOM_STATE)
plt.scatter(sample["Historical_Stop_Mean"], sample[TARGET], alpha=0.25, label="Training observations")
plt.plot(x_grid["Historical_Stop_Mean"], linear_fit.predict(x_grid), label="Simple Linear Regression")
plt.plot(x_grid["Historical_Stop_Mean"], poly_fit.predict(x_grid), label="Polynomial Regression (degree 2)")
plt.title("Figure 16. Linear vs Polynomial Fit")
plt.xlabel("Historical_Stop_Mean")
plt.ylabel("Passenger_Count")
plt.legend()
plt.tight_layout()
plt.show()

## 13. Select the best single-feature model

Selection is based primarily on chronological test RMSE, with MAE and R² as supporting measures. Time-series cross-validation is used as an additional stability check. The selected model is not assumed in advance; the notebook selects it from the measured results.

In [ ]:
best_model_name = comparison_df.iloc[0]["Model"]
best_test_model = fitted_models[best_model_name]
best_test_pred = predictions[best_model_name]
best_test_metrics = regression_metrics(y_test, best_test_pred)

print("Selected single-feature model:", best_model_name)
print("Test metrics:")
for metric, value in best_test_metrics.items():
    print(f"{metric}: {value:.4f}")

best_cv_row = cv_df.loc[cv_df["Model"] == best_model_name].iloc[0]
print("\nTime-series CV RMSE mean:", round(best_cv_row["CV_RMSE_Mean"], 4))
print("Time-series CV RMSE std :", round(best_cv_row["CV_RMSE_Std"], 4))

In [ ]:
# Figure 17: Best model detailed diagnostic dashboard
resid = y_test.to_numpy() - np.asarray(best_test_pred)

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
sns.scatterplot(x=y_test, y=best_test_pred, alpha=0.5, ax=axes[0, 0])
lo = min(float(y_test.min()), float(np.min(best_test_pred)))
hi = max(float(y_test.max()), float(np.max(best_test_pred)))
axes[0, 0].plot([lo, hi], [lo, hi], linestyle="--")
axes[0, 0].set_title("Actual vs Predicted")

sns.histplot(resid, kde=True, bins=25, ax=axes[0, 1])
axes[0, 1].set_title("Residual Distribution")

sns.scatterplot(x=best_test_pred, y=resid, alpha=0.5, ax=axes[1, 0])
axes[1, 0].axhline(0, linestyle="--")
axes[1, 0].set_title("Residuals vs Predicted")

sns.scatterplot(x=np.arange(len(y_test)), y=resid, alpha=0.5, ax=axes[1, 1])
axes[1, 1].axhline(0, linestyle="--")
axes[1, 1].set_title("Residuals in Test-Period Order")
axes[1, 1].set_xlabel("Test row order")

fig.suptitle(f"Figure 17. Detailed Diagnostics — {best_model_name}", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 14. Final model training and export

The holdout metrics above are the honest estimate of performance. After the model has been selected and evaluated, we can fit a final model on all available labelled records for future use. The holdout scores must remain reported as the evaluation results; do not claim that the refitted model was independently tested on the same rows it was trained on.

In [ ]:
# Rebuild the Historical_Stop_Mean feature over the full time-ordered labelled dataset.
final_data = df.copy().sort_values("_timestamp").reset_index(drop=True)

final_data["_stop_prior_mean"] = (
    final_data.groupby(STOP_COL, sort=False)[TARGET]
              .transform(lambda s: s.shift(1).expanding(min_periods=1).mean())
)
final_data["_global_prior_mean"] = final_data[TARGET].shift(1).expanding(min_periods=1).mean()
final_data["Historical_Stop_Mean"] = final_data["_stop_prior_mean"].combine_first(
    final_data["_global_prior_mean"]
)
final_data = final_data.dropna(subset=["Historical_Stop_Mean"]).copy()

# Refit the selected model type on all rows with a valid historical feature.
final_X = final_data[["Historical_Stop_Mean"]]
final_y = final_data[TARGET].astype(float)

final_model = clone(single_feature_models[best_model_name])
final_model.fit(final_X, final_y)

MODEL_FILE = "best_passenger_count_model.joblib"
joblib.dump({
    "model": final_model,
    "model_name": best_model_name,
    "feature_columns": ["Historical_Stop_Mean"],
    "target": TARGET,
    "evaluation_metrics": best_test_metrics,
    "course_code": "24CSE0316"
}, MODEL_FILE)

print("Final model saved:", MODEL_FILE)

In [ ]:
# 15. Export result tables and test predictions
comparison_df.to_csv("single_feature_model_comparison.csv", index=False)
cv_df.to_csv("time_series_cross_validation.csv", index=False)

test_predictions_df = test[["_timestamp", STOP_COL, TARGET, "Historical_Stop_Mean"]].copy()
test_predictions_df["Predicted_Passenger_Count"] = best_test_pred
test_predictions_df["Residual"] = (
    test_predictions_df[TARGET] - test_predictions_df["Predicted_Passenger_Count"]
)
test_predictions_df.to_csv("best_model_test_predictions.csv", index=False)

summary = {
    "selected_model": best_model_name,
    "test_metrics": {k: float(v) for k, v in best_test_metrics.items()},
    "training_rows": int(len(train)),
    "testing_rows": int(len(test)),
    "test_period_start": str(test["_timestamp"].min()),
    "test_period_end": str(test["_timestamp"].max()),
    "single_feature": "Historical_Stop_Mean",
    "cross_validation_rmse_mean": float(best_cv_row["CV_RMSE_Mean"]),
    "cross_validation_rmse_std": float(best_cv_row["CV_RMSE_Std"])
}
with open("evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=4)

display(test_predictions_df.head(10))
print(json.dumps(summary, indent=4))

In [ ]:
# 16. Save all currently generated figures and output files in one ZIP.
# Re-run this cell after all figures have been generated.
OUTPUT_DIR = Path("passenger_prediction_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Save tables and model files.
for file_name in [
    "single_feature_model_comparison.csv",
    "time_series_cross_validation.csv",
    "best_model_test_predictions.csv",
    "evaluation_summary.json",
    MODEL_FILE
]:
    if Path(file_name).exists():
        import shutil
        shutil.copy(file_name, OUTPUT_DIR / file_name)

# Save all open Matplotlib figures if any remain open.
for i, fig_num in enumerate(plt.get_fignums(), start=1):
    plt.figure(fig_num).savefig(OUTPUT_DIR / f"figure_open_{i:02d}.png", dpi=300, bbox_inches="tight")

zip_name = "passenger_prediction_project_outputs.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUTPUT_DIR))

print("Created:", zip_name)
files.download(zip_name)

## 17. Interpretation checklist for the project report

After running the notebook, record:

1. Actual usable dataset dimensions and any rows excluded for invalid required fields.
2. Chronological train/test sizes and the date boundary.
3. The number of initial training rows without historical target information.
4. All four test metrics for each single-feature model.
5. Time-series cross-validation RMSE mean and standard deviation.
6. Whether training and validation errors suggest overfitting or underfitting.
7. Multiple Linear Regression results as a separate feature-set experiment.
8. The selected model and the reason for selecting it.
9. The synthetic-data limitation.

**Do not copy the earlier report's model scores into the final report until this notebook has been run on the actual final CSV.** The new results may differ because the historical feature calculation is explicitly leakage-safe.